# Modul 2: Backpropagation dan Automatic Differentiation

**Nama:** Ahmad Rizqi  
**NIM:**  122450138
**Kelas:** RA  
**Tanggal:** 2026-09-27  

Simpan berkas ini sebagai `M02_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan.
2. Nilai Kasus 1 tidak boleh diubah; seluruh angka pada modul mengacu padanya.
3. Gunakan `float64` untuk semua perhitungan gradien.
4. Tuliskan turunan manual pada sel markdown, bukan hanya di kertas.
5. Notebook harus lolos *Restart Kernel and Run All* sebelum dikumpulkan.
6. Luaran: `M02_NIM.ipynb`, `M02_NIM.pdf`, dan `M02_NIM_metrics.csv`.

In [1]:
import platform
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch import nn

NIM = '122450138' 
SEED = int(str(NIM)[-9:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
torch.set_default_dtype(torch.float64)
pd.set_option('display.precision', 8)
environment = {
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': torch.__version__,
    'device': str(DEVICE),
    'seed': SEED,
    'default_dtype': str(torch.get_default_dtype()),
}
environment

{'python': '3.11.4',
 'numpy': '2.4.6',
 'pandas': '3.0.6',
 'torch': '2.14.0+cpu',
 'device': 'cpu',
 'seed': 122450138,
 'default_dtype': 'torch.float64'}

## A. Pre-lab - 10 poin

Jawab sebelum sesi praktikum dimulai.

1. **Gradien lokal vs gradien total pada satu simpul:** TODO
2. **Mengapa `backward()` hanya dapat dipanggil pada tensor skalar:** TODO
3. **Isi `.grad` bila `backward()` dipanggil dua kali tanpa `zero_grad()`:** TODO
4. **Mengapa turunan BCE-with-logits terhadap logit berbentuk $p-y$, bukan $-y/p$:** TODO

**Graf komputasi Kasus 1.** Tuliskan urutan simpul dari $\mathbf{x}$ sampai $\mathcal{L}$, lalu tandai gradien lokal di setiap simpul:

TODO

## B. Turunan manual - 20 poin

Kasus 1 memakai nilai tetap berikut:

$$\mathbf{x}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
\mathbf{W}^{(1)}=\begin{bmatrix}0.5 & -0.5\\ 1 & 1\end{bmatrix},\quad
\mathbf{b}^{(1)}=\begin{bmatrix}0 & 0\end{bmatrix},\quad
\mathbf{W}^{(2)}=\begin{bmatrix}2 & -1\end{bmatrix},\quad
b^{(2)}=0.5,\quad y=1$$

Tuliskan penurunan Anda **berurutan** di sini, satu baris satu langkah:

1. $\partial\mathcal{L}/\partial z^{(2)} =$ TODO
2. $\partial\mathcal{L}/\partial \mathbf{W}^{(2)} =$ TODO
3. $\partial\mathcal{L}/\partial b^{(2)} =$ TODO
4. $\partial\mathcal{L}/\partial \mathbf{h} =$ TODO
5. $\partial\mathcal{L}/\partial \mathbf{z}^{(1)} =$ TODO
6. $\partial\mathcal{L}/\partial \mathbf{W}^{(1)} =$ TODO
7. $\partial\mathcal{L}/\partial \mathbf{b}^{(1)} =$ TODO

Cantumkan pula shape setiap gradien: TODO

In [2]:
x = np.array([2.0, -1.0], dtype=np.float64)
W1 = np.array([[0.5, -0.5], [1.0, 1.0]], dtype=np.float64)
b1 = np.array([0.0, 0.0], dtype=np.float64)
W2 = np.array([2.0, -1.0], dtype=np.float64)
b2 = np.float64(0.5)
y = np.float64(1.0)

def sigmoid_stabil(z):
    z = np.asarray(z, dtype=np.float64)
    return np.where(z >= 0, 1.0 / (1.0 + np.exp(-z)), np.exp(z) / (1.0 + np.exp(z)))

def forward(x, W1, b1, W2, b2, y):
    z1 = W1 @ x + b1
    h = np.maximum(z1, 0.0)
    z2 = W2 @ h + b2
    p = sigmoid_stabil(z2).item()
    # Bentuk BCE-with-logits yang stabil: softplus(z2) - y*z2.
    loss = np.logaddexp(0.0, z2) - y * z2
    return {'z1': z1, 'h': h, 'z2': z2, 'p': p, 'loss': loss}

nilai = forward(x, W1, b1, W2, b2, y)
print({k: np.round(v, 6) for k, v in nilai.items()})

# Pemeriksaan wajib: tidak diubah.
assert np.allclose(nilai['z1'], [1.5, 1.0]), 'z1 belum benar'
assert np.isclose(nilai['z2'], 2.5), 'logit belum benar'
assert np.isclose(nilai['loss'], 0.0788897, atol=1e-6), 'loss belum benar'
print('forward pass sesuai Kasus 1')

{'z1': array([1.5, 1. ]), 'h': array([1.5, 1. ]), 'z2': np.float64(2.5), 'p': np.float64(0.924142), 'loss': np.float64(0.07889)}
forward pass sesuai Kasus 1


In [3]:
def backward(x, W1, W2, y, nilai):
    z1, h, p = nilai['z1'], nilai['h'], nilai['p']
    dz2 = p - y
    dW2 = dz2 * h
    db2 = dz2
    dh = dz2 * W2
    dz1 = dh * (z1 > 0)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

grad_manual = backward(x, W1, W2, y, nilai)
for nama, value in grad_manual.items():
    print(f'{nama:>3}: {np.round(value, 7)}')


assert np.allclose(grad_manual['W2'], [-0.1137873, -0.0758582], atol=1e-6)
assert grad_manual['W1'].shape == W1.shape, 'shape dW1 harus sama dengan W1'
print('gradien manual sesuai angka acuan')

 W1: [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]
 b1: [-0.1517164  0.0758582]
 W2: [-0.1137873 -0.0758582]
 b2: -0.0758582
gradien manual sesuai angka acuan


## C. Autograd - 20 poin

Bangun ulang Kasus 1 dengan tensor PyTorch, lalu bandingkan gradiennya dengan hasil bagian B.

In [4]:
tW1 = torch.tensor(W1, requires_grad=True)
tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True)
tb2 = torch.tensor(b2, requires_grad=True)
tx, ty = torch.tensor(x), torch.tensor(y)
kriteria = nn.BCEWithLogitsLoss()

def forward_torch():
    tz1 = tW1 @ tx + tb1
    th = torch.relu(tz1)
    tz2 = tW2 @ th + tb2
    return kriteria(tz2, ty)

loss = forward_torch()
loss.backward()

for nama, tensor in [('W1', tW1), ('b1', tb1), ('W2', tW2), ('b2', tb2)]:
    selisih = np.max(np.abs(tensor.grad.detach().numpy() - grad_manual[nama]))
    print(f'{nama}: autograd = {np.round(tensor.grad.detach().numpy(), 7)}   selisih maks = {selisih:.2e}')
    assert selisih < 1e-10, f'gradien {nama} tidak sesuai dengan hasil manual'
print('autograd cocok dengan backward manual')

W1: autograd = [[-0.3034327  0.1517164]
 [ 0.1517164 -0.0758582]]   selisih maks = 0.00e+00
b1: autograd = [-0.1517164  0.0758582]   selisih maks = 0.00e+00
W2: autograd = [-0.1137873 -0.0758582]   selisih maks = 0.00e+00
b2: autograd = -0.0758582   selisih maks = 0.00e+00
autograd cocok dengan backward manual


In [5]:
grad_W2_satu = tW2.grad.detach().clone()
forward_torch().backward()  
grad_W2_dua = tW2.grad.detach().clone()
print('satu backward        :', np.round(grad_W2_satu.numpy(), 7))
print('dua backward         :', np.round(grad_W2_dua.numpy(), 7))
assert torch.allclose(grad_W2_dua, 2 * grad_W2_satu)

for tensor in (tW1, tb1, tW2, tb2):
    tensor.grad = None
forward_torch().backward()
grad_W2_reset = tW2.grad.detach().clone()
print('setelah grad dihapus :', np.round(grad_W2_reset.numpy(), 7))
assert torch.allclose(grad_W2_reset, grad_W2_satu)

satu backward        : [-0.1137873 -0.0758582]
dua backward         : [-0.2275745 -0.1517164]
setelah grad dihapus : [-0.1137873 -0.0758582]


**Penjelasan akumulasi gradien:** Ketika `backward()` dijalankan untuk kedua kalinya tanpa mengosongkan nilai `.grad`, diperoleh `tW2.grad = [-0.2275745, -0.1517164]`. Nilai tersebut merupakan **dua kali lipat** dari gradien yang dihasilkan oleh satu kali proses `backward()`, yaitu `[-0.1137873, -0.0758582]`. Setelah atribut `.grad` diatur menjadi `None`, proses `backward()` berikutnya kembali menghasilkan nilai gradien seperti semula. Hal ini menunjukkan bahwa PyTorch secara default melakukan akumulasi gradien. Oleh karena itu, karena optimizer menggunakan nilai yang tersimpan pada `.grad` ketika menjalankan `step()`, proses training umumnya diawali dengan `optimizer.zero_grad()` sebelum melakukan perhitungan backward pada batch berikutnya. Dengan demikian, gradien dari batch atau iterasi sebelumnya tidak tercampur dengan gradien yang sedang dihitung.


## D. Gradient checking - 20 poin

Bandingkan gradien analitik dengan selisih terpusat:

$$g_\text{num}=\frac{\mathcal{L}(\theta+\epsilon)-\mathcal{L}(\theta-\epsilon)}{2\epsilon},
\qquad
\text{rel err}=\frac{|g_\text{analitik}-g_\text{num}|}{|g_\text{analitik}|+|g_\text{num}|+10^{-12}}$$

Ambang lulus: seluruh baris di bawah $10^{-5}$.

In [6]:
EPS = 1e-5

def loss_dengan(param, i, delta):
    parameters = {
        'W1': W1.copy(),
        'b1': b1.copy(),
        'W2': W2.copy(),
        'b2': np.float64(b2),
    }
    if param == 'b2':
        parameters[param] += delta
    else:
        parameters[param][i] += delta
    return forward(x, parameters['W1'], parameters['b1'],
                   parameters['W2'], parameters['b2'], y)['loss']

def finite_difference(param, i, epsilon=EPS):
    return (loss_dengan(param, i, epsilon) - loss_dengan(param, i, -epsilon)) / (2 * epsilon)

indeks = ([('W1', (0, 0)), ('W1', (0, 1)), ('W1', (1, 0)), ('W1', (1, 1))]
          + [('b1', (0,)), ('b1', (1,))]
          + [('W2', (0,)), ('W2', (1,))]
          + [('b2', ())])

baris = []
for nama, i in indeks:
    manual = grad_manual[nama][i] if i != () else grad_manual[nama]
    auto_array = {'W1': tW1, 'b1': tb1, 'W2': tW2, 'b2': tb2}[nama].grad.detach().numpy()
    auto = auto_array[i] if i != () else auto_array
    numerik = finite_difference(nama, i)
    rel = abs(manual - numerik) / (abs(manual) + abs(numerik) + 1e-12)
    baris.append({'parameter': nama, 'indeks': str(i), 'manual': manual,
                  'autograd': float(auto), 'numerik': numerik, 'rel_err': rel})

tabel = pd.DataFrame(baris)
display(tabel)
print('relative error maksimum:', tabel['rel_err'].max())
assert len(tabel) == 9, 'tabel harus memuat sembilan komponen parameter'
assert tabel['rel_err'].max() < 1e-5, 'masih ada baris yang melampaui ambang'

,parameter,indeks,manual,autograd,numerik,rel_err
0,W1,"(0, 0)",-0.30343272,-0.30343272,-0.30343272,1.07313401e-10
1,W1,"(0, 1)",0.15171636,0.15171636,0.15171636,3.41358596e-11
2,W1,"(1, 0)",0.15171636,0.15171636,0.15171636,3.41358596e-11
3,W1,"(1, 1)",-0.07585818,-0.07585818,-0.07585818,1.12219224e-10
4,b1,"(0,)",-0.15171636,-0.15171636,-0.15171636,3.41358596e-11
5,b1,"(1,)",0.07585818,0.07585818,0.07585818,1.12219224e-10
6,W2,"(0,)",-0.11378727,-0.11378727,-0.11378727,8.29208263e-11
7,W2,"(1,)",-0.07585818,-0.07585818,-0.07585818,1.12219224e-10
8,b2,(),-0.07585818,-0.07585818,-0.07585818,1.12219224e-10


relative error maksimum: 1.1221922358447185e-10


In [7]:
tabel_gradcheck = tabel.copy()
tabel_gradcheck.insert(0, 'run_id', 'gradcheck_kasus1')
tabel_gradcheck.insert(1, 'record_type', 'gradient_check')
tabel_gradcheck.insert(2, 'seed', SEED)
tabel_gradcheck.to_csv(f'M02_{NIM}_metrics.csv', index=False)
print(f'M02_{NIM}_metrics.csv tersimpan (sementara; akan digabung dengan hasil berikutnya).')

M02_122450138_metrics.csv tersimpan (sementara; akan digabung dengan hasil berikutnya).


**Checkpoint menit ke-95.** Tunjukkan tabel sembilan baris di atas kepada asisten sebelum melanjutkan ke bagian E.

## E. Diagnosis training loop - 20 poin

Fungsi `train_rusak` di bawah berjalan **tanpa pesan galat**, tetapi memuat **empat** kesalahan. Kasus yang dipakai adalah XOR dengan protokol modul: FNN $2 \rightarrow 4 \rightarrow 1$, SGD `lr=0.1`, 400 epoch, satu batch penuh.

In [8]:
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([[0.0], [1.0], [1.0], [0.0]])

def train_rusak(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.Linear(4, 1),                
    )
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    kriteria = nn.BCEWithLogitsLoss()
    riwayat = []

    for _ in range(epoch):
        logits = model(X_xor)
        loss = kriteria(torch.sigmoid(logits), y_xor)  
        opt.step()                                    

       
        logits_for_backward = model(X_xor)
        loss_for_backward = kriteria(torch.sigmoid(logits_for_backward), y_xor)
        loss_for_backward.backward()                 
        riwayat.append(loss.item())
    return model, riwayat

model_rusak, riwayat_rusak = train_rusak()
print(f'loss awal  : {riwayat_rusak[0]:.4f}')
print(f'loss akhir : {riwayat_rusak[-1]:.4f}')
with torch.no_grad():
    print('prediksi   :', (torch.sigmoid(model_rusak(X_xor)) > 0.5).int().flatten().tolist())
    print('target     :', y_xor.int().flatten().tolist())

loss awal  : 0.7315
loss akhir : 0.5995
prediksi   : [0, 0, 1, 0]
target     : [0, 1, 1, 0]


### Temuan Kesalahan

| No | Bagian kode yang bermasalah                       | Alasan kesalahan                                                                                                                                                                                                                           | Dampak yang terlihat                                                                                                                                  |
| -: | ------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------- |
|  1 | `nn.Linear(2, 4), nn.Linear(4, 1)`                | Kedua layer tersebut tidak disertai fungsi aktivasi nonlinier. Akibatnya, rangkaian layer affine tersebut secara matematis setara dengan satu layer affine sehingga tidak mampu mempelajari pola XOR.                                      | Nilai loss sulit mencapai kondisi yang menunjukkan pemisahan XOR secara tepat.                                                                        |
|  2 | `kriteria(torch.sigmoid(logits), y_xor)`          | `BCEWithLogitsLoss` dirancang untuk menerima nilai logit secara langsung karena proses sigmoid sudah dilakukan secara internal.                                                                                                            | Fungsi objektif menjadi tidak sesuai dan gradien yang dihasilkan dapat menjadi lebih kecil sehingga proses pembelajaran berjalan lebih lambat.        |
|  3 | `opt.step()` dijalankan sebelum `loss.backward()` | Parameter model diperbarui sebelum gradien dari loss pada iterasi tersebut dihitung. Akibatnya, pada iterasi awal tidak terjadi pembaruan yang sesuai, sedangkan iterasi berikutnya dapat menggunakan gradien dari perhitungan sebelumnya. | Proses pembaruan parameter menjadi terlambat satu iterasi dan tidak secara langsung merepresentasikan loss yang baru dihitung.                        |
|  4 | Tidak terdapat `opt.zero_grad()`                  | PyTorch secara default mengakumulasi gradien pada atribut `.grad` setiap kali `backward()` dipanggil, bukan menggantinya dengan nilai baru.                                                                                                | Gradien dari iterasi sebelumnya terus terbawa sehingga nilainya dapat semakin besar dan menyebabkan proses pembaruan parameter menjadi kurang stabil. |

**Catatan versi:** Pada PyTorch 2.7, kode starter yang dijalankan tanpa perubahan dapat menghasilkan error *in-place version mismatch* pada iterasi kedua karena `step()` melakukan perubahan terhadap parameter di antara proses forward dan backward. Dalam eksperimen, graf komputasi dibuat kembali setelah langkah yang keliru sehingga pengaruh penggunaan gradien sebelumnya masih dapat diamati sekaligus memastikan notebook dapat dijalankan sampai selesai.


In [9]:
def evaluasi_xor(model):
    model.eval()
    with torch.no_grad():
        logits = model(X_xor)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_xor).item()
        predictions = (torch.sigmoid(logits) > 0.5).to(torch.int64).flatten()
        correct = int((predictions == y_xor.to(torch.int64).flatten()).sum().item())
    return loss, correct, predictions.tolist()

def train_varian(*, use_activation, use_raw_logits, correct_order, clear_gradients,
                  epoch=400, lr=0.1):
    seed_everything(SEED)
    layers = [nn.Linear(2, 4)]
    if use_activation:
        layers.append(nn.ReLU())
    layers.append(nn.Linear(4, 1))
    model = nn.Sequential(*layers)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    initial_bce, _, _ = evaluasi_xor(model)

    for _ in range(epoch):
        if clear_gradients:
            optimizer.zero_grad()
        logits = model(X_xor)
        loss_input = logits if use_raw_logits else torch.sigmoid(logits)
        loss = criterion(loss_input, y_xor)
        if correct_order:
            loss.backward()
            optimizer.step()
        else:
            optimizer.step()
            # Bangun ulang graf setelah langkah keliru agar PyTorch modern
            # tidak menghentikan eksperimen akibat in-place version mismatch.
            logits_after_step = model(X_xor)
            loss_input_after_step = (logits_after_step if use_raw_logits
                                     else torch.sigmoid(logits_after_step))
            criterion(loss_input_after_step, y_xor).backward()
        history.append(loss.item())
    return model, history, initial_bce

konfigurasi_tahap = [
    ('rusak', 'belum ada perbaikan', False, False, False, False),
    ('perbaikan-1', 'menambahkan ReLU', True, False, False, False),
    ('perbaikan-2', 'memberikan logit mentah ke loss', True, True, False, False),
    ('perbaikan-3', 'memindahkan backward sebelum optimizer.step', True, True, True, False),
    ('perbaikan-4', 'menambahkan optimizer.zero_grad', True, True, True, True),
]

tahap = []
model_tahap = {}
for nama_tahap, perbaikan, activation_ok, logits_ok, order_ok, zero_ok in konfigurasi_tahap:
    model, history, initial_bce = train_varian(
        use_activation=activation_ok,
        use_raw_logits=logits_ok,
        correct_order=order_ok,
        clear_gradients=zero_ok,
    )
    final_bce, correct, predictions = evaluasi_xor(model)
    model_tahap[nama_tahap] = model
    tahap.append({
        'run_id': f'xor_{nama_tahap}',
        'tahap': nama_tahap,
        'yang_diperbaiki': perbaikan,
        'loss_awal': initial_bce,
        'loss_akhir': final_bce,
        'objective_akhir': history[-1],
        'benar_dari_4': correct,
        'prediksi': str(predictions),
        'epochs': 400,
    })

df_tahap_preview = pd.DataFrame(tahap)
display(df_tahap_preview)

,run_id,tahap,yang_diperbaiki,loss_awal,loss_akhir,objective_akhir,benar_dari_4,prediksi,epochs
0,xor_rusak,rusak,belum ada perbaikan,0.70594475,6.72584915e+01,5.99458203e-01,3,"[0, 0, 1, 0]",400
1,xor_perbaikan-1,perbaikan-1,menambahkan ReLU,0.69020792,8.12209695e+01,6.93147181e-01,2,"[0, 0, 0, 0]",400
2,xor_perbaikan-2,perbaikan-2,memberikan logit mentah ke loss,0.69020792,2.56340362e-27,3.03166390e-27,4,"[0, 1, 1, 0]",400
3,xor_perbaikan-3,perbaikan-3,memindahkan backward sebelum optimizer.step,0.69020792,2.16746921e-27,2.56340362e-27,4,"[0, 1, 1, 0]",400
4,xor_perbaikan-4,perbaikan-4,menambahkan optimizer.zero_grad,0.69020792,1.72369662e-01,1.72655009e-01,4,"[0, 1, 1, 0]",400


In [ ]:
def train_benar(epoch: int = 400, lr: float = 0.1):
    seed_everything(SEED)
    model = nn.Sequential(
        nn.Linear(2, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []

    for _ in range(epoch):
        optimizer.zero_grad()
        logits = model(X_xor)
        loss = criterion(logits, y_xor)
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history

model_benar, riwayat_benar = train_benar()
print(f'loss awal : {riwayat_benar[0]:.4f}')
print(f'loss akhir: {riwayat_benar[-1]:.4f}')
with torch.no_grad():
    prediksi = (torch.sigmoid(model_benar(X_xor)) > 0.5).int().flatten()
print('prediksi  :', prediksi.tolist())

assert riwayat_benar[-1] < 0.1, 'loss akhir harus di bawah 0,1'
assert torch.equal(prediksi, y_xor.int().flatten()), 'keempat titik XOR harus benar'
print('training loop sudah benar')

loss awal : 0.6902
loss akhir: 0.1727
prediksi  : [0, 1, 1, 0]


AssertionError: loss akhir harus di bawah 0,1

In [11]:
df_tahap = pd.DataFrame(tahap)
df_tahap.insert(1, 'record_type', 'training_loop')
df_tahap.insert(2, 'seed', SEED)
df_tahap.to_csv(f'M02_{NIM}_metrics_loop.csv', index=False)

# Gabungan sementara; tugas individu akan menambahkan gradient check perluasan.
metrics_combined = pd.concat([tabel_gradcheck, df_tahap], ignore_index=True, sort=False)
metrics_combined.to_csv(f'M02_{NIM}_metrics.csv', index=False)
display(df_tahap)

,run_id,record_type,seed,tahap,yang_diperbaiki,loss_awal,loss_akhir,objective_akhir,benar_dari_4,prediksi,epochs
0,xor_rusak,training_loop,122450138,rusak,belum ada perbaikan,0.70594475,6.72584915e+01,5.99458203e-01,3,"[0, 0, 1, 0]",400
1,xor_perbaikan-1,training_loop,122450138,perbaikan-1,menambahkan ReLU,0.69020792,8.12209695e+01,6.93147181e-01,2,"[0, 0, 0, 0]",400
2,xor_perbaikan-2,training_loop,122450138,perbaikan-2,memberikan logit mentah ke loss,0.69020792,2.56340362e-27,3.03166390e-27,4,"[0, 1, 1, 0]",400
3,xor_perbaikan-3,training_loop,122450138,perbaikan-3,memindahkan backward sebelum optimizer.step,0.69020792,2.16746921e-27,2.56340362e-27,4,"[0, 1, 1, 0]",400
4,xor_perbaikan-4,training_loop,122450138,perbaikan-4,menambahkan optimizer.zero_grad,0.69020792,1.72369662e-01,1.72655009e-01,4,"[0, 1, 1, 0]",400


## F. Tugas individu

Kerjakan ketiganya di sel-sel baru di bawah bagian ini.

1. **Perluasan jaringan.** Tambahkan neuron ketiga pada hidden layer: baris $[-1\;\;0.5]$ pada $\mathbf{W}^{(1)}$, bias $0{,}25$, dan komponen $-0{,}5$ pada $\mathbf{W}^{(2)}$. Turunkan manual, implementasikan, lalu buat tabel relative error yang baru.
2. **Batch dua contoh.** Tambahkan $\mathbf{x}_2=[-1\;\;3]$ dengan $y_2=0$, pakai rata-rata loss, dan jelaskan di langkah mana gradien kedua contoh dijumlahkan.
3. **Laporan diagnosis.** Rangkum keempat kesalahan beserta bukti angka sebelum dan sesudah setiap perbaikan.

## G. Pertanyaan analisis

1. Mengapa relative error tidak pernah persis nol, dan berapa nilai yang masih wajar? TODO
2. Apa yang terjadi pada tabel bila $\epsilon = 10^{-9}$? Jalankan dan jelaskan. TODO
3. Pada langkah mana gradien contoh pertama dan kedua bergabung saat memakai batch? TODO
4. Kesalahan mana pada bagian E yang paling sulit ditemukan tanpa membandingkan angka? TODO
5. Apa beda peran backpropagation dan optimizer? (maksimal tiga kalimat) TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, dan device tercantum.
- [ ] Seluruh `TODO` dan `raise NotImplementedError` sudah diganti.
- [ ] Turunan manual ditulis pada sel markdown bagian B.
- [ ] Tabel relative error memuat sembilan baris dan seluruhnya lulus ambang.
- [ ] Keempat kesalahan bagian E ditemukan, dibuktikan, dan diperbaiki bertahap.
- [ ] Notebook lolos *Restart Kernel and Run All*.
- [ ] Berkas: `M02_NIM.ipynb`, `M02_NIM.pdf`, `M02_NIM_metrics.csv`.